<a href="https://colab.research.google.com/github/NagaRaju1991/google_colab_notebooks/blob/fsds_course/03_XOR_Pytorch_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install onnxscript


### Import all the libraries

In [ ]:
import torch
import torch.nn as nn
import torch.onnx
import onnxscript
from onnxscript import ir

### Step 1 : Intialize the data for XOR Gate

In [ ]:
# 1. SETUP DATA
# Inputs: [0,0], [0,1], [1,0], [1,1]
inputs = torch.tensor([[0,0], [0,1], [1,0], [1,1]], dtype=torch.float32)
# XOR Target: [[0],[1],[1],[0]]
target = torch.tensor([[0], [1], [1], [0]], dtype=torch.float32)

### Step 2 : Define the model

In [ ]:
# 2. DEFINE THE MODEL
model = nn.Sequential(
    nn.Linear(2, 4),    # 2 inputs to 4 hidden nodes
    nn.ReLU(),          # Activation
    nn.Linear(4, 1),    # 4 hidden nodes to 1 output
    nn.Sigmoid()        # Output stays between 0 and 1
)

### Step 3 : Train the model

In [ ]:
# 3. TRAINING LOOP
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.BCELoss()

print("Training... please wait.")
for epoch in range(500):
    pred = model(inputs)            # Forward Pass
    loss = loss_fn(pred, target)    # Calculate Error

    optimizer.zero_grad()           # Clear old math
    loss.backward()                 # Backpropagation
    optimizer.step()                # Update Weights

Training... please wait.


### Step 4 : Display the results

In [ ]:
print("\n--- Final Predictions ---")
with torch.no_grad():
    output = (model(inputs) > 0.5).float()
    for i in range(4):
        print(f"In: {inputs[i].tolist()} -> Out: {output[i].item()}")


--- Final Predictions ---
In: [0.0, 0.0] -> Out: 0.0
In: [0.0, 1.0] -> Out: 1.0
In: [1.0, 0.0] -> Out: 1.0
In: [1.0, 1.0] -> Out: 0.0


### Step 5 : Inspect the weights of the model

In [ ]:
# 5. MANUAL WEIGHT INSPECTION
print("\n--- Manual Weight Inspection ---")
# state_dict() contains all learnable parameters (weights and biases)
params = model.state_dict()
for name, param in params.items():
    print(f"\nLayer: {name}")
    print(param.numpy())


--- Manual Weight Inspection ---

Layer: 0.weight
[[ 2.4491358  -2.298608  ]
 [ 3.1760707  -3.1764028 ]
 [-0.22926587 -0.6977602 ]
 [ 3.1782007  -3.1779969 ]]

Layer: 0.bias
[ 2.2969215e+00 -2.3290335e-04 -1.6259538e-01 -4.7783699e-04]

Layer: 2.weight
[[-5.4881673   4.911296    0.48518336  4.1366057 ]]

Layer: 2.bias
[5.444192]


#### Step 6 : Export the model to ONNX format

In [ ]:
# 6. EXPORT TO ONNX
# ONNX requires a 'dummy input' to trace the graph structure
dummy_input = torch.tensor([[0.0, 0.0]], dtype=torch.float32)
onnx_filename = "XOR_logic_gate_model.onnx"

torch.onnx.export(
    model,               # The trained model
    dummy_input,         # Sample input for tracing
    onnx_filename,       # Output filename
    export_params=True,  # Store the trained weight values inside the file
    opset_version=11,    # ONNX version to use
    input_names=['input'],   # Label for input node
    output_names=['output']  # Label for output node
)

print(f"\nModel successfully saved to: {onnx_filename}")

/tmp/ipykernel_275/1039170184.py:6: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
W0309 10:28:38.468000 275 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅

Model successfully saved to: XOR_logic_gate_model.onnx
